# CP1 Week 13 -- Integration: End-to-End Pipeline

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Create plots with matplotlib
2. Build a `plot()` function for the pipeline
3. Run all 5 pipeline stages end-to-end
4. Generate required figures (timeseries + summary)

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Matplotlib Basics

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

# Simple line plot
data = [10, 15, 13, 18, 20, 17, 22, 25, 23, 28]

plt.figure(figsize=(10, 4))
plt.plot(data, marker="o", color="steelblue")
plt.title("My First Plot")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.grid(True, alpha=0.3)
os.makedirs("reports/figures", exist_ok=True)
plt.savefig("reports/figures/test_plot.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved!")

**Expected Output:** A line plot with blue markers showing values increasing
from 10 to 28 over 10 time steps, with gridlines and labels.

### Anatomy of a plot

```
  +------ Title -------+
  |  ^                  |
  |  | Y-axis label     |
  |  |    * --- *       |
  |  |   / \   / \      |
  |  |  *   *     *     |
  |  +--+---+---+---+-> |
  |     X-axis label    |
  +---------------------+
```

Every plot needs: **title**, **x-label**, **y-label**, **grid** (for readability).
Missing any of these is a common mistake.

### Example 2 -- Customizing plots

In [ ]:
import random
random.seed(42)

# Generate two series
temps = [20 + random.gauss(0, 5) for _ in range(30)]
rpms = [1500 + random.gauss(0, 200) for _ in range(30)]

# Plot with customization
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(temps, color="red", linewidth=1.5, marker=".", label="Temperature")
ax.set_title("Motor Temperature Over Time")
ax.set_xlabel("Reading Number")
ax.set_ylabel("Temperature (C)")
ax.grid(True, alpha=0.3)
ax.axhline(y=30, color="orange", linestyle="--", label="Warning Threshold")
ax.legend()
plt.tight_layout()
plt.savefig("reports/figures/custom_plot.png", dpi=100, bbox_inches="tight")
plt.show()
print("Custom plot saved!")

---
## Part 3: Multiple Subplots

In [ ]:
import random
random.seed(42)
values = [random.gauss(50, 15) for _ in range(200)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(values, linewidth=0.8, color="steelblue")
ax1.set_title("Time Series")
ax1.set_xlabel("Index")
ax1.set_ylabel("Value")
ax1.grid(True, alpha=0.3)

ax2.hist(values, bins=20, color="steelblue", edgecolor="white")
ax2.set_title("Distribution")
ax2.set_xlabel("Value")
ax2.set_ylabel("Count")
mean_val = sum(values) / len(values)
ax2.axvline(mean_val, color="red", label="Mean")
ax2.legend()

plt.tight_layout()
plt.savefig("reports/figures/combined.png", dpi=100)
plt.show()
print("Combined plot saved!")

---
## Part 3: Pipeline plot() Function

In [ ]:
def plot(clean_data, results, config):
    """Create and save required figures."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    values = [row["value"] for row in clean_data if "value" in row]
    if not values:
        print("No data to plot")
        return []

    fig_dir = config.get("figures_dir", "reports/figures")
    os.makedirs(fig_dir, exist_ok=True)
    figures = []

    # Figure 1: Time Series
    fig1, ax = plt.subplots(figsize=(10, 4))
    ax.plot(values, linewidth=0.8, color="steelblue")
    title = config.get("project_name", "Project") + " -- Time Series"
    ax.set_title(title)
    ax.set_xlabel("Index")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)

    threshold = config.get("threshold")
    if threshold:
        label_text = "Threshold=" + str(threshold)
        ax.axhline(y=threshold, color="red", linestyle="--", label=label_text)
        ax.legend()

    path1 = os.path.join(fig_dir, "timeseries.png")
    fig1.savefig(path1, dpi=100, bbox_inches="tight")
    figures.append(path1)
    plt.close(fig1)
    print(f"Saved: {path1}")

    # Figure 2: Summary histogram
    fig2, ax = plt.subplots(figsize=(8, 4))
    n_bins = min(20, max(5, len(values) // 5))
    ax.hist(values, bins=n_bins, color="steelblue", edgecolor="white")
    ax.set_title("Value Distribution")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

    summary = results.get("analysis_summary", {})
    if "mean" in summary:
        label_text = "Mean=" + str(summary["mean"])
        ax.axvline(summary["mean"], color="red", label=label_text)
        ax.legend()

    path2 = os.path.join(fig_dir, "summary.png")
    fig2.savefig(path2, dpi=100, bbox_inches="tight")
    figures.append(path2)
    plt.close(fig2)
    print(f"Saved: {path2}")

    return figures

# Test
test_data = [{"value": random.gauss(50, 10)} for _ in range(100)]
test_results = {"analysis_summary": {"mean": 50}}
test_config = {"project_name": "Test", "figures_dir": "reports/figures", "threshold": 65}
plot(test_data, test_results, test_config)

---
## Part 4: Running the Complete Pipeline

Now let us connect ALL 5 stages and run them end-to-end:

In [ ]:
import csv, json, os, random

# --- CONFIG ---
config = {
    "project_name": "integration_test",
    "track": "test",
    "version": "v1",
    "n_points": 50,
    "min_value": 0,
    "max_value": 100,
    "threshold": 60,
    "cleaned_data_path": "data/cleaned/cleaned.csv",
    "report_path": "reports/report.json",
    "figures_dir": "reports/figures",
}

# --- STEP 1: LOAD ---
def load_data(config):
    random.seed(42)
    data = []
    for i in range(config.get("n_points", 50)):
        val = random.gauss(40, 20)
        data.append({"index": i, "value": str(round(val, 2))})
    # Add some bad data
    data.append({"index": 50, "value": ""})
    data.append({"index": 51, "value": "abc"})
    data.append({"index": 52, "value": "999"})
    print(f"Loaded {len(data)} rows")
    return data

# --- STEP 2: CLEAN ---
def clean_data(data, config):
    cleaned = []
    dropped = 0
    for row in data:
        try:
            val = float(row["value"])
            if config["min_value"] <= val <= config["max_value"]:
                cleaned.append({**row, "value": val})
            else:
                dropped += 1
        except (ValueError, TypeError):
            dropped += 1
    print(f"Cleaned: {len(data)} -> {len(cleaned)} ({dropped} dropped)")
    return cleaned

# --- STEP 3: ANALYZE ---
def analyze(cleaned, config):
    values = [r["value"] for r in cleaned]
    n = len(values)
    mean_val = sum(values) / n if n else 0
    sorted_v = sorted(values)
    median_val = sorted_v[n // 2] if n else 0
    variance = sum((x - mean_val)**2 for x in values) / n if n else 0
    std_val = variance ** 0.5
    above = sum(1 for v in values if v > config.get("threshold", 60))
    results = {
        "project_name": config["project_name"],
        "track": config["track"],
        "version": config["version"],
        "dataset": {"n_raw": len(cleaned) + 5, "n_clean": n},
        "analysis_summary": {
            "count": n, "mean": round(mean_val, 2),
            "median": round(median_val, 2), "std": round(std_val, 2),
            "min": round(min(values), 2) if values else 0,
            "max": round(max(values), 2) if values else 0,
            "above_threshold": above,
        },
        "figures": ["timeseries.png", "summary.png"],
    }
    print(f"Analyzed: {n} values, mean={mean_val:.2f}, std={std_val:.2f}")
    return results

# --- RUN IT ---
print("=== FULL PIPELINE RUN ===")
print()
data = load_data(config)
cleaned = clean_data(data, config)
results = analyze(cleaned, config)
figures = plot(cleaned, results, config)

# --- STEP 5: EXPORT ---
os.makedirs(os.path.dirname(config["cleaned_data_path"]), exist_ok=True)
if cleaned:
    keys = list(cleaned[0].keys())
    with open(config["cleaned_data_path"], "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(cleaned)
    print(f"Exported: {config['cleaned_data_path']}")

os.makedirs(os.path.dirname(config["report_path"]), exist_ok=True)
with open(config["report_path"], "w") as f:
    json.dump(results, f, indent=2)
print(f"Exported: {config['report_path']}")

print()
print("=== PIPELINE COMPLETE ===")

### Verifying all outputs exist

In [ ]:
# Quick verification
import os
required = [
    "data/cleaned/cleaned.csv",
    "reports/report.json",
    "reports/figures/timeseries.png",
    "reports/figures/summary.png",
]

print("=== Output Verification ===")
all_ok = True
for path in required:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        size = os.path.getsize(path)
        print(f"  [OK] {path} ({size} bytes)")
    else:
        print(f"  [MISSING] {path}")
        all_ok = False

print()
if all_ok:
    print("All outputs verified!")
else:
    print("Some outputs missing -- check your pipeline.")

---
## Key Takeaways -- Week 13

1. **matplotlib** creates publication-quality plots
2. **`plt.savefig()`** saves plots to files (required for your pipeline)
3. Your **`plot()`** function must create timeseries.png and summary.png
4. Everything connects: load -> clean -> analyze -> **plot** -> export
5. **Verification** after running ensures nothing was missed

### Try It Yourself

In [ ]:
# TODO: Create a bar chart showing label counts.
# Given these labels, create a bar chart:

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

labels_data = ["normal"] * 30 + ["warning"] * 12 + ["critical"] * 5

# Count each label
counts = {}
for label in labels_data:
    counts[label] = counts.get(label, 0) + 1

# Create bar chart
# plt.figure(figsize=(8, 4))
# plt.bar(counts.keys(), counts.values(), color=["green", "orange", "red"])
# plt.title("Alert Distribution")
# plt.ylabel("Count")
# plt.savefig("reports/figures/alerts.png")
# plt.show()
print("TODO: Uncomment the code above to create the bar chart")
print(f"Counts: {counts}")

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What is matplotlib.pyplot?
# R2: What does plt.savefig() do?
# R3: Why do we call plt.close(fig) after saving?
# R4: What is a subplot?

### Practice (P1-P5)

In [ ]:
# P1: Create a scatter plot of two related variables.
# Example: temperature vs RPM


In [ ]:
# P2: Create a bar chart showing counts by category.


In [ ]:
# P3: Run YOUR complete pipeline end-to-end and verify all exports.


In [ ]:
# P4: Add a third figure type to your plot() function.


In [ ]:
# P5: Create a 2x2 subplot dashboard of your data.


### Challenge (C1-C2)

In [ ]:
# C1: Create a plot that shows data points colored by label.
# Normal = blue, Warning = orange, Critical = red


In [ ]:
# C2: Create an animated-looking plot by plotting cumulative data
# at different lengths (save multiple PNGs).


### Mini-Project

In [ ]:
# M1: Complete Pipeline Run
# 1. Load -> Clean -> Analyze -> Plot -> Export
# 2. Verify: cleaned.csv, report.json, timeseries.png, summary.png
# 3. Print a summary of what was created
# 4. Re-read the exports and verify they match


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)